# Notebook 11 — Performance Improvement Experiments
## FYP: Adaptive Explainable Ensemble for Pre-Launch Steam Game Reception Prediction
### Heshan Ratnaweera | IIT Sri Lanka | W2082289 | 2026

**Purpose:** Test four research-backed approaches to improve Macro F1 beyond
the current best of 0.6796 (uncertainty router, NB04 FINAL).

**Experiments:**
1. CatBoost auto_class_weights='Balanced' (built-in calibrated weighting)
2. Weighted Cross-Entropy via per-sample weights (research-recommended for GBDT imbalance)
3. Label Smoothing (softens hard 0/1 labels to reduce boundary noise)
4. Optuna Bayesian hyperparameter optimisation (smarter than RandomizedSearchCV)
5. Gated Text Fusion (structured features gate SBERT dims — fixes Model E's core failure)

**Nothing is saved to disk. All results stay inside this notebook.**

**Inputs (read-only):**
- `data/processed/games_features_t4.csv`
- `data/processed/sbert_embeddings_pca50.npy`

---
## Contents
1. Setup and Imports
2. Configuration and Baselines
3. Load Data and Define Features
4. Experiment 1 — Auto Class Weights
5. Experiment 2 — Weighted Cross-Entropy
6. Experiment 3 — Label Smoothing
7. Experiment 4 — Optuna Bayesian HPO
8. Experiment 5 — Gated Text Fusion
9. Overall Summary and Verdict

## 1. Setup and Imports

In [ ]:
import sys
import warnings
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.linear_model import Ridge
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    TARGET_COL, COL_DESC, RANDOM_STATE,
    TOP_10_GENRES, TOP_20_TAGS, KEY_CATEGORIES
)

# ── Install Optuna if not present ─────────────────────────────────────────────
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    print('Optuna already installed ✓')
except ImportError:
    print('Installing Optuna...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'optuna', '--quiet'])
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    print('Optuna installed ✓')

print('All imports OK')

## 2. Configuration and Baselines

In [ ]:
# ── Base CatBoost params — identical to NB04 FINAL ───────────────────────────
CB_BASE = {
    'depth': 8,
    'learning_rate': 0.05,
    'iterations': 1000,
    'l2_leaf_reg': 3,
    'scale_pos_weight': 0.5,
    'random_seed': RANDOM_STATE,
    'verbose': 0,
    'allow_writing_files': False
}

# ── Baselines from NB04 FINAL (direct routing, held-out test set) ─────────────
BASELINE_MACRO   = 0.6690
BASELINE_MINOR   = 0.5172
BASELINE_MAJOR   = 0.8209
BASELINE_ACC     = 0.7385
BASELINE_AUC     = 0.7383
BASELINE_ROUTER  = 0.6796   # uncertainty router — the best overall result

# ── Cross-validation settings (same as all previous notebooks) ────────────────
CV_FOLDS  = 5
TEST_SIZE = 0.20

# Class ratio for manual weight calculation
POS_RATE  = 0.718          # 71.8% positive (Well Received)
NEG_RATE  = 0.282          # 28.2% negative (Not Well Received)
POS_WEIGHT = NEG_RATE / POS_RATE   # ~0.393 — mathematically optimal WCE weight

print(f'Base params       : depth={CB_BASE["depth"]}, lr={CB_BASE["learning_rate"]}, '
      f'iter={CB_BASE["iterations"]}, l2={CB_BASE["l2_leaf_reg"]}')
print(f'Current baseline  : Macro F1 = {BASELINE_MACRO}  |  Router = {BASELINE_ROUTER}')
print(f'WCE pos_weight    : {POS_WEIGHT:.4f}  (ratio of minority to majority rate)')

## 3. Load Data and Define Features

In [ ]:
# ── Load structured dataset ───────────────────────────────────────────────────
df = pd.read_csv(PROJECT_ROOT / 'data' / 'processed' / 'games_features_t4.csv')
print(f'Dataset: {len(df):,} rows x {df.shape[1]} columns')
print(f'Class balance: {df[TARGET_COL].value_counts(normalize=True).round(3).to_dict()}')

# ── Load SBERT embeddings ─────────────────────────────────────────────────────
embeddings = np.load(PROJECT_ROOT / 'data' / 'processed' / 'sbert_embeddings_pca50.npy')
print(f'SBERT embeddings: {embeddings.shape}')
assert embeddings.shape[0] == len(df), 'Row count mismatch between dataset and embeddings'

# ── Feature columns ───────────────────────────────────────────────────────────
EXCLUDE     = [TARGET_COL, COL_DESC]
T4_FEATURES = [c for c in df.columns if c not in EXCLUDE]
TEXT_DIMS   = [f'text_dim_{i}' for i in range(50)]
print(f'T4 structured features: {len(T4_FEATURES)}')

# ── Train / test split — same as all previous notebooks ──────────────────────
y   = df[TARGET_COL].values
idx = np.arange(len(df))

(X_train_df, X_test_df,
 idx_train,  idx_test,
 y_train,    y_test) = train_test_split(
    df[T4_FEATURES], idx, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

X_train = X_train_df.values
X_test  = X_test_df.values
emb_train = embeddings[idx_train]
emb_test  = embeddings[idx_test]

print(f'Train: {len(y_train):,} games  ({y_train.mean()*100:.1f}% positive)')
print(f'Test : {len(y_test):,} games  ({y_test.mean()*100:.1f}% positive)')

In [ ]:
# ── Shared evaluation helper ─────────────────────────────────────────────────
def evaluate(model, X_te, y_te, label=''):
    """Evaluate a trained model on the test set and return a results dict."""
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]
    r = {
        'label':    label,
        'macro':    round(f1_score(y_te, y_pred, average='macro'), 4),
        'minor':    round(f1_score(y_te, y_pred, pos_label=0),     4),
        'major':    round(f1_score(y_te, y_pred, pos_label=1),     4),
        'accuracy': round(accuracy_score(y_te, y_pred),            4),
        'auc':      round(roc_auc_score(y_te, y_prob),             4),
    }
    if label:
        print(f'{label}:')
        print(f'  Macro F1    : {r["macro"]:.4f}  (baseline {BASELINE_MACRO},'
              f' change {r["macro"]-BASELINE_MACRO:+.4f})')
        print(f'  Minority F1 : {r["minor"]:.4f}  (baseline {BASELINE_MINOR},'
              f' change {r["minor"]-BASELINE_MINOR:+.4f})')
        print(f'  Accuracy    : {r["accuracy"]:.4f}  (baseline {BASELINE_ACC},'
              f' change {r["accuracy"]-BASELINE_ACC:+.4f})')
        print(f'  AUC-ROC     : {r["auc"]:.4f}')
    return r

results_log = []    # accumulate all experiment results for final summary

## 4. Experiment 1 — CatBoost Auto Class Weights

**What this tests:** CatBoost's built-in `auto_class_weights='Balanced'` which
automatically computes the mathematically optimal weight from the training data's
class distribution. Currently the project uses a manually set `scale_pos_weight=0.5`.

**Why it might help:** The manual 0.5 was found by tuning in NB03. The auto-balanced
version computes the exact inverse frequency ratio (~0.393 for this dataset) which is
the theoretically correct weight for cross-entropy under class imbalance.

In [ ]:
print('Training Model D with auto_class_weights=Balanced...')

CB_AUTO = {k: v for k, v in CB_BASE.items() if k != 'scale_pos_weight'}
CB_AUTO['auto_class_weights'] = 'Balanced'

m_auto = CatBoostClassifier(**CB_AUTO)
m_auto.fit(X_train, y_train)
r_auto = evaluate(m_auto, X_test, y_test, 'Experiment 1 — Auto Class Weights')
results_log.append(r_auto)

## 5. Experiment 2 — Weighted Cross-Entropy via Sample Weights

**What this tests:** Per-sample weighting — every positive (Well Received) game gets
weight `w_pos = NEG_RATE / POS_RATE ≈ 0.393`, every negative gets weight 1.0.
Passing `sample_weight` to `.fit()` is mathematically equivalent to Weighted
Cross-Entropy and is the method peer-reviewed research (Luo, Yuan & Xu, arXiv:2407.14381)
found "consistently provides the most substantial gains" on GBDT macro-F1 for
imbalanced datasets.

**Why this differs from scale_pos_weight=0.5:** The current 0.5 was found by grid search,
not by the theoretically optimal formula. The WCE weight (0.393) is derived directly
from the class ratio and is the loss-minimising solution.

In [ ]:
print('Training Model D with Weighted Cross-Entropy (sample_weight)...')
print(f'WCE pos_weight = {POS_WEIGHT:.4f}  neg_weight = 1.0000')
print()

# ── Build per-sample weight vector ────────────────────────────────────────────
sample_weights = np.where(y_train == 1, POS_WEIGHT, 1.0)

# ── Remove scale_pos_weight — sample_weight handles imbalance instead ─────────
CB_WCE = {k: v for k, v in CB_BASE.items() if k != 'scale_pos_weight'}

m_wce = CatBoostClassifier(**CB_WCE)
m_wce.fit(X_train, y_train, sample_weight=sample_weights)
r_wce = evaluate(m_wce, X_test, y_test, 'Experiment 2 — Weighted Cross-Entropy')
results_log.append(r_wce)

# ── Also test two alternative weight ratios around the optimal ────────────────
print()
print('Testing alternative weight ratios:')
for w in [0.3, 0.393, 0.45, 0.5]:
    sw = np.where(y_train == 1, w, 1.0)
    m  = CatBoostClassifier(**CB_WCE)
    m.fit(X_train, y_train, sample_weight=sw)
    y_pred = m.predict(X_test)
    macro  = f1_score(y_test, y_pred, average='macro')
    minor  = f1_score(y_test, y_pred, pos_label=0)
    print(f'  pos_weight={w:.3f}  Macro F1={macro:.4f}  Minority F1={minor:.4f}')

## 6. Experiment 3 — Label Smoothing

**What this tests:** Instead of hard 0/1 binary labels, slightly soften the targets
so the model learns appropriate uncertainty about borderline games near the 75%
threshold. A game at 74% is labelled 0.05 instead of 0 — the model learns to be
uncertain about it rather than confidently wrong.

**Why it might help:** Games at 73-77% positive reviews are genuinely ambiguous —
their hard binary label (0 or 1) is essentially noise at the boundary. ICML 2020
research showed label smoothing is competitive with explicit noise-correction methods.

CatBoost's `CrossEntropy` loss accepts float targets between 0 and 1, making this
implementable without a custom loss function.

In [ ]:
print('Training with Label Smoothing (epsilon variants)...')
print()

CB_LS = {k: v for k, v in CB_BASE.items() if k != 'scale_pos_weight'}
CB_LS['loss_function'] = 'CrossEntropy'   # accepts float targets

best_ls_result = None
best_ls_macro  = 0.0

for epsilon in [0.0, 0.05, 0.10, 0.15]:
    # Smooth labels: 1 → (1 - epsilon), 0 → epsilon
    smooth_targets = np.where(y_train == 1, 1.0 - epsilon, epsilon)

    m_ls = CatBoostClassifier(**CB_LS)
    m_ls.fit(X_train, smooth_targets)

    y_pred = m_ls.predict(X_test)
    macro  = f1_score(y_test, y_pred, average='macro')
    minor  = f1_score(y_test, y_pred, pos_label=0)
    acc    = accuracy_score(y_test, y_pred)

    tag = ' ← baseline (no smoothing)' if epsilon == 0 else ''
    print(f'  epsilon={epsilon:.2f}  Macro F1={macro:.4f}  '
          f'Minority F1={minor:.4f}  Accuracy={acc:.4f}{tag}')

    if macro > best_ls_macro:
        best_ls_macro  = macro
        best_ls_result = {
            'label':    f'Experiment 3 — Label Smoothing (eps={epsilon})',
            'macro':    round(macro, 4),
            'minor':    round(minor, 4),
            'major':    round(f1_score(y_test, y_pred, pos_label=1), 4),
            'accuracy': round(acc, 4),
            'auc':      round(roc_auc_score(y_test, m_ls.predict_proba(X_test)[:, 1]), 4),
        }

print()
print(f'Best label smoothing result: {best_ls_result["label"]}')
print(f'  Macro F1={best_ls_result["macro"]:.4f}  '
      f'Minority F1={best_ls_result["minor"]:.4f}  '
      f'Accuracy={best_ls_result["accuracy"]:.4f}')
results_log.append(best_ls_result)

## 7. Experiment 4 — Optuna Bayesian Hyperparameter Optimisation

**What this tests:** Optuna's Tree-structured Parzen Estimator (TPE) — a Bayesian
optimisation algorithm that learns which hyperparameter regions perform best and
focuses search there. Unlike NB03's RandomizedSearchCV which picks 30 random points,
Optuna's 50 trials are increasingly informed by previous results.

**Key difference from NB03:** The objective function optimises **Macro F1 directly**
using 3-fold CV (for speed), searching depth, learning_rate, iterations, l2_leaf_reg,
and scale_pos_weight simultaneously.

In [ ]:
print('Running Optuna Bayesian HPO (50 trials, optimising Macro F1)...')
print('This will take several minutes — Optuna will print progress every 10 trials.')
print()

skf_optuna = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

def optuna_objective(trial):
    params = {
        'depth':             trial.suggest_int('depth', 6, 10),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'iterations':        trial.suggest_int('iterations', 600, 1500),
        'l2_leaf_reg':       trial.suggest_int('l2_leaf_reg', 1, 10),
        'scale_pos_weight':  trial.suggest_float('scale_pos_weight', 0.3, 0.7),
        'random_seed':       RANDOM_STATE,
        'verbose':           0,
        'allow_writing_files': False,
    }

    fold_f1s = []
    for tr_idx, val_idx in skf_optuna.split(X_train, y_train):
        m = CatBoostClassifier(**params)
        m.fit(X_train[tr_idx], y_train[tr_idx])
        pred = m.predict(X_train[val_idx])
        fold_f1s.append(f1_score(y_train[val_idx], pred, average='macro'))

    return np.mean(fold_f1s)

study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)
study.optimize(optuna_objective, n_trials=50, show_progress_bar=True)

best_params = study.best_params
best_cv_f1  = study.best_value

print(f'\nOptuna best CV Macro F1 : {best_cv_f1:.4f}')
print(f'Best hyperparameters:')
for k, v in best_params.items():
    original = CB_BASE.get(k, 'N/A')
    change   = ' ← CHANGED' if str(v) != str(original) else ' (same)'
    print(f'  {k:<22}: {v}{change}')

In [ ]:
# ── Train final model with Optuna best params and evaluate on test set ────────
print('\nTraining final model with Optuna best params...')

CB_OPTUNA = {**best_params,
             'random_seed': RANDOM_STATE,
             'verbose': 0,
             'allow_writing_files': False}

m_optuna = CatBoostClassifier(**CB_OPTUNA)
m_optuna.fit(X_train, y_train)
r_optuna = evaluate(m_optuna, X_test, y_test, 'Experiment 4 — Optuna HPO')
results_log.append(r_optuna)

## 8. Experiment 5 — Gated Text Fusion

**What this tests:** A structured-feature gate that suppresses SBERT text dimensions
which merely restate what the structured features already say, and amplifies the
dimensions that carry genuinely independent signal.

**Why naive concatenation failed (NB05):** Model E concatenated all 50 SBERT dims
with all 53 structured features. The structured features were always stronger so the
model learned to ignore the text completely. The gate fixes this by:

1. For each of the 50 SBERT dimensions, fit a Ridge regression using the 53 structured
   features to predict that text dimension
2. Gate value = (1 - R²) — how much of that dimension is NOT explained by structured features
3. Multiply each SBERT dimension by its gate value
4. Gate → 0 means "structured features already know this" → suppress it
5. Gate → 1 means "this text dimension carries new information" → keep it

The gated text features are then concatenated with the structured features and trained
as a new Model E equivalent.

**Research basis:** Gameiro (2025) SFL architecture and Arevalo et al. (2017) Gated
Multimodal Units both use this principle of structured features modulating text.

In [ ]:
print('Computing structured-feature gate for SBERT dimensions...')
print()

# ── Step 1: Fit one Ridge regression per SBERT dim using structured features ──
gate_values = np.zeros(50)
r2_scores   = np.zeros(50)

for dim_i in range(50):
    ridge = Ridge(alpha=1.0)
    ridge.fit(X_train, emb_train[:, dim_i])
    r2 = ridge.score(X_train, emb_train[:, dim_i])
    r2_scores[dim_i]  = max(r2, 0.0)
    gate_values[dim_i] = 1.0 - max(r2, 0.0)   # independent signal = 1 - R²

# ── Report gate statistics ────────────────────────────────────────────────────
print(f'Gate analysis across 50 SBERT dimensions:')
print(f'  Mean gate value : {gate_values.mean():.4f}  '
      f'(1.0 = fully independent, 0.0 = fully explained by structured)')
print(f'  Min gate value  : {gate_values.min():.4f}  (most redundant dim)')
print(f'  Max gate value  : {gate_values.max():.4f}  (most independent dim)')
print(f'  Dims with gate > 0.8 (high independence): '
      f'{(gate_values > 0.8).sum()}')
print(f'  Dims with gate < 0.3 (mostly redundant) : '
      f'{(gate_values < 0.3).sum()}')
print()

# Show top 5 most independent and most redundant dims
top5_ind = np.argsort(gate_values)[-5:][::-1]
top5_red = np.argsort(gate_values)[:5]
print('  Most independent SBERT dims (gate closest to 1.0):')
for i in top5_ind:
    print(f'    text_dim_{i:>2}  gate={gate_values[i]:.4f}  R²={r2_scores[i]:.4f}')
print('  Most redundant SBERT dims (gate closest to 0.0):')
for i in top5_red:
    print(f'    text_dim_{i:>2}  gate={gate_values[i]:.4f}  R²={r2_scores[i]:.4f}')

In [ ]:
# ── Step 2: Apply gate to SBERT embeddings ───────────────────────────────────
gated_train = emb_train * gate_values   # element-wise: each dim scaled by its gate
gated_test  = emb_test  * gate_values

# ── Step 3: Concatenate gated text with structured features ───────────────────
X_tr_gated = np.hstack([X_train, gated_train])   # (16306, 103)
X_te_gated = np.hstack([X_test,  gated_test])    # (4077, 103)

print(f'Gated feature matrix shape: {X_tr_gated.shape}')
print()

# ── Step 4: Train CatBoost on gated features ──────────────────────────────────
print('Training Model E with gated text fusion...')

CB_GATED = {**CB_BASE}   # same params as base — fair comparison
m_gated  = CatBoostClassifier(**CB_GATED)
m_gated.fit(X_tr_gated, y_train)
r_gated  = evaluate(m_gated, X_te_gated, y_test, 'Experiment 5 — Gated Text Fusion')
results_log.append(r_gated)

print()

# ── Compare against original naive concatenation (NB05 result) ────────────────
ORIGINAL_MODEL_E_MACRO = 0.6622   # from NB05 FINAL
print(f'Original Model E (naive concat): Macro F1 = {ORIGINAL_MODEL_E_MACRO}')
print(f'Gated fusion improvement        : {r_gated["macro"] - ORIGINAL_MODEL_E_MACRO:+.4f}')

## 9. Overall Summary and Verdict

In [ ]:
print('=' * 82)
print('NB11 OVERALL SUMMARY')
print('=' * 82)
print()
print(f'  {"Approach":<42} {"Macro F1":>10} {"Minor F1":>10} {"Accuracy":>10} {"vs Baseline":>12}')
print('  ' + '-' * 88)

# Baseline rows
print(f'  {"Baseline: Model D direct (NB04 FINAL)":<42} {BASELINE_MACRO:>10.4f} '
      f'{BASELINE_MINOR:>10.4f} {BASELINE_ACC:>10.4f} {"—":>12}')
print(f'  {"Baseline: Uncertainty router (NB04 FINAL)":<42} {BASELINE_ROUTER:>10.4f} '
      f'{"—":>10} {"—":>10} {"—":>12}')
print('  ' + '- ' * 44)

best_new_macro  = 0.0
best_new_result = None

for r in results_log:
    diff = r['macro'] - BASELINE_MACRO
    diff_str = f'{diff:+.4f}'
    print(f'  {r["label"]:<42} {r["macro"]:>10.4f} '
          f'{r["minor"]:>10.4f} {r["accuracy"]:>10.4f} {diff_str:>12}')
    if r['macro'] > best_new_macro:
        best_new_macro  = r['macro']
        best_new_result = r

print()
print('=' * 82)
print('VERDICT')
print('=' * 82)
print()

if best_new_macro > BASELINE_ROUTER + 0.003:
    print(f'CLEAR WIN: {best_new_result["label"]}')
    print(f'  Macro F1 = {best_new_macro:.4f}  —  beats BOTH baselines')
    print(f'  Improvement over direct routing : {best_new_macro - BASELINE_MACRO:+.4f}')
    print(f'  Improvement over router         : {best_new_macro - BASELINE_ROUTER:+.4f}')
    print()
    print('Recommendation: Integrate this approach into the main system (NB03-NB07 rerun).')

elif best_new_macro > BASELINE_MACRO + 0.003:
    print(f'MARGINAL WIN over direct routing: {best_new_result["label"]}')
    print(f'  Macro F1 = {best_new_macro:.4f}  (direct baseline {BASELINE_MACRO})')
    print(f'  Does not beat the uncertainty router ({BASELINE_ROUTER})')
    print()
    print('Recommendation: Consider integrating if the improvement is consistent across')
    print('multiple runs. The uncertainty router remains the overall champion.')

elif best_new_macro >= BASELINE_MACRO - 0.002:
    print(f'NO MEANINGFUL CHANGE: best new result = {best_new_macro:.4f}')
    print(f'  Within noise of the {BASELINE_MACRO} baseline.')
    print()
    print('Recommendation: Keep the existing system. Document these experiments as')
    print('a thorough exploration in the thesis — each negative result is a finding.')

else:
    print(f'ALL APPROACHES DEGRADED performance below baseline ({BASELINE_MACRO}).')
    print(f'Best was {best_new_macro:.4f} from: {best_new_result["label"]}')
    print()
    print('Recommendation: Keep the original system unchanged.')
    print('The current architecture is already optimal for this dataset and feature set.')

print()
print('Individual findings:')
for r in results_log:
    diff = r['macro'] - BASELINE_MACRO
    tag  = ' ✓ IMPROVEMENT' if diff > 0.003 else (' ~ marginal' if diff > 0 else ' ✗ worse')
    print(f'  {r["label"]:<42} {diff:+.4f}{tag}')